## 🎯 Learning Objectives
* Understand the specialized roles of sub-agents within a multi-agent system for complex tasks.
* Implement dedicated sub-agents for hotel reservation, availability checking, and modification using modern agent frameworks.
* Integrate and utilize custom tools effectively within agent definitions to interact with external systems.
* Define clear and concise roles, goals, and backstories for agents to guide their behavior and decision-making.


## Exercise: Building the Reservation, Availability, and Modification Sub-Agents

In this exercise, you will implement the core sub-agents responsible for handling specific aspects of hotel bookings: checking availability, making reservations, and modifying existing reservations. These agents will be crucial components of our multi-agent hotel reservation system, each equipped with specialized tools to perform their designated tasks.

### Task Description
Your task is to define three distinct `crewai.Agent` instances:
1.  **`AvailabilityAgent`**: Responsible for checking hotel availability based on user criteria.
2.  **`ReservationAgent`**: Responsible for making new hotel reservations.
3.  **`ModificationAgent`**: Responsible for modifying or canceling existing hotel reservations.

### Requirements
*   Each agent must be an instance of `crewai.Agent`.
*   Each agent must have a unique and descriptive `role`, `goal`, and `backstory` that clearly defines its purpose.
*   Each agent must be assigned specific tools relevant to its function. You will use the mock tools provided in the setup code.
*   Set `verbose=True` for all agents to observe their thought process during execution.
*   Set `allow_delegation=False` for these specific agents, as they are designed to perform their own specialized tasks directly.

### Evaluation Criteria
*   **Correct Agent Instantiation**: All three agents (`AvailabilityAgent`, `ReservationAgent`, `ModificationAgent`) are correctly instantiated as `crewai.Agent` objects.
*   **Appropriate Roles, Goals, and Backstories**: The `role`, `goal`, and `backstory` for each agent are logical, distinct, and align with their intended function.
*   **Correct Tool Assignment**: Each agent is assigned the correct mock tool(s) that enable it to perform its task.
*   **Configuration**: `verbose` is set to `True` and `allow_delegation` is set to `False` for all agents.
*   **Code Quality**: The code is clean, well-structured, and includes comments where necessary to explain design choices.


In [ ]:
import os
from crewai import Agent, Tool
from typing import Dict, Any

# Mock Hotel Data Store (in a real system, this would be a database or external API)
mock_hotel_data = {
    "hotel_paradise": {
        "rooms_available": {
            "standard": 10,
            "deluxe": 5,
            "suite": 2
        },
        "bookings": {}
    },
    "grand_hotel": {
        "rooms_available": {
            "standard": 15,
            "executive": 8
        },
        "bookings": {}
    }
}

# --- Mock Tools Definition ---
# In a real-world scenario, these would be wrappers around actual APIs (e.g., REST, gRPC)

class HotelAvailabilityTool(Tool):
    name: str = "HotelAvailabilityChecker"
    description: str = "Checks the availability of rooms for a specific hotel and room type."

    def _run(self, hotel_name: str, room_type: str) -> str:
        """Checks room availability."""
        hotel_name_lower = hotel_name.lower().replace(' ', '_')
        room_type_lower = room_type.lower()

        if hotel_name_lower not in mock_hotel_data:
            return f"Error: Hotel '{hotel_name}' not found."

        hotel = mock_hotel_data[hotel_name_lower]
        if room_type_lower not in hotel["rooms_available"]:
            return f"Error: Room type '{room_type}' not available at '{hotel_name}'."

        available_rooms = hotel["rooms_available"][room_type_lower]
        return f"There are {available_rooms} '{room_type}' rooms available at '{hotel_name}'."

class HotelBookingTool(Tool):
    name: str = "HotelReservationMaker"
    description: str = "Makes a new hotel reservation for a specified hotel, room type, and guest details."

    def _run(self, hotel_name: str, room_type: str, guest_name: str, check_in_date: str, check_out_date: str) -> str:
        """Makes a hotel reservation."""
        hotel_name_lower = hotel_name.lower().replace(' ', '_')
        room_type_lower = room_type.lower()

        if hotel_name_lower not in mock_hotel_data:
            return f"Error: Hotel '{hotel_name}' not found."

        hotel = mock_hotel_data[hotel_name_lower]
        if room_type_lower not in hotel["rooms_available"] or hotel["rooms_available"][room_type_lower] <= 0:
            return f"Error: No '{room_type}' rooms available at '{hotel_name}'."

        # Simulate booking
        hotel["rooms_available"][room_type_lower] -= 1
        booking_id = f"BOOK-{len(hotel['bookings']) + 1:04d}"
        hotel["bookings"][booking_id] = {
            "guest_name": guest_name,
            "room_type": room_type,
            "check_in": check_in_date,
            "check_out": check_out_date,
            "status": "confirmed"
        }
        return f"Reservation '{booking_id}' confirmed for {guest_name} at '{hotel_name}' for a '{room_type}' room from {check_in_date} to {check_out_date}."

class HotelModificationTool(Tool):
    name: str = "HotelReservationModifier"
    description: "Modifies or cancels an existing hotel reservation using a booking ID."

    def _run(self, booking_id: str, new_check_in: str = None, new_check_out: str = None, new_room_type: str = None, action: str = "modify") -> str:
        """Modifies or cancels a hotel reservation."""
        found_booking = None
        hotel_name_found = None

        for h_name, h_data in mock_hotel_data.items():
            if booking_id in h_data["bookings"]:
                found_booking = h_data["bookings"][booking_id]
                hotel_name_found = h_name
                break

        if not found_booking:
            return f"Error: Booking ID '{booking_id}' not found."

        if action == "cancel":
            original_room_type = found_booking["room_type"].lower()
            mock_hotel_data[hotel_name_found]["rooms_available"][original_room_type] += 1
            found_booking["status"] = "cancelled"
            return f"Booking '{booking_id}' for {found_booking['guest_name']} has been cancelled."
        elif action == "modify":
            updates = []
            if new_check_in and new_check_in != found_booking["check_in"]:
                found_booking["check_in"] = new_check_in
                updates.append(f"check-in to {new_check_in}")
            if new_check_out and new_check_out != found_booking["check_out"]:
                found_booking["check_out"] = new_check_out
                updates.append(f"check-out to {new_check_out}")
            if new_room_type and new_room_type.lower() != found_booking["room_type"].lower():
                # Simulate room type change - release old, try to book new
                original_room_type = found_booking["room_type"].lower()
                new_room_type_lower = new_room_type.lower()

                if new_room_type_lower not in mock_hotel_data[hotel_name_found]["rooms_available"] or \
                   mock_hotel_data[hotel_name_found]["rooms_available"][new_room_type_lower] <= 0:
                    return f"Modification failed: New room type '{new_room_type}' not available."

                mock_hotel_data[hotel_name_found]["rooms_available"][original_room_type] += 1
                mock_hotel_data[hotel_name_found]["rooms_available"][new_room_type_lower] -= 1
                found_booking["room_type"] = new_room_type
                updates.append(f"room type to {new_room_type}")

            if updates:
                return f"Booking '{booking_id}' for {found_booking['guest_name']} modified: {', '.join(updates)}."
            else:
                return f"No changes requested for booking '{booking_id}'."
        else:
            return f"Invalid action '{action}'. Must be 'modify' or 'cancel'."

# Instantiate the mock tools
availability_tool = HotelAvailabilityTool()
booking_tool = HotelBookingTool()
modification_tool = HotelModificationTool()

print("Mock tools and data initialized successfully!")
print(f"Initial availability for Hotel Paradise (standard): {mock_hotel_data['hotel_paradise']['rooms_available']['standard']}")


### Your Implementation

Now, it's your turn to define the three sub-agents: `AvailabilityAgent`, `ReservationAgent`, and `ModificationAgent`. Use the `crewai.Agent` class and assign the appropriate mock tools provided in the setup cell.

Remember to set `verbose=True` and `allow_delegation=False` for each agent.


In [ ]:
# Define the Availability Agent
availability_agent = Agent(
    role='Hotel Availability Specialist',
    goal='Provide accurate and up-to-date information on hotel room availability.',
    backstory=(
        "You are an expert in hotel inventory management. Your primary function is to query "
        "the hotel system for room availability based on specific criteria like hotel name "
        "and room type. You ensure that all availability checks are precise and reflect "
        "the current state of the hotel's inventory."
    ),
    tools=[availability_tool], # Assign the availability checking tool
    verbose=True,
    allow_delegation=False
)

# Define the Reservation Agent
reservation_agent = Agent(
    role='Hotel Booking Coordinator',
    goal='Secure new hotel reservations efficiently and accurately.',
    backstory=(
        "You are a meticulous booking agent responsible for processing new hotel "
        "reservations. You take guest details, desired hotel, room type, and dates, "
        "and use the booking system to confirm the reservation. Your focus is on "
        "accuracy and ensuring a smooth booking experience."
    ),
    tools=[booking_tool], # Assign the booking tool
    verbose=True,
    allow_delegation=False
)

# Define the Modification Agent
modification_agent = Agent(
    role='Reservation Adjustment Specialist',
    goal='Handle modifications and cancellations of existing hotel reservations.',
    backstory=(
        "You are a flexible and detail-oriented agent specializing in changes to "
        "existing bookings. Whether it's updating dates, changing room types, or "
        "canceling a reservation, you use the modification system to implement "
        "these changes precisely and communicate the outcome."
    ),
    tools=[modification_tool], # Assign the modification tool
    verbose=True,
    allow_delegation=False
)

print("\n--- Agents Defined Successfully ---")
print(f"Availability Agent Role: {availability_agent.role}")
print(f"Reservation Agent Role: {reservation_agent.role}")
print(f"Modification Agent Role: {modification_agent.role}")

# Example of how an agent might use its tool (for demonstration, not part of the exercise output)
# This would typically be orchestrated by a Crew or a higher-level agent.
# print("\n--- Testing Availability Agent (Manual Call) ---")
# availability_result = availability_agent.tools[0]._run(hotel_name="Hotel Paradise", room_type="standard")
# print(f"Availability Agent Test Result: {availability_result}")

# print("\n--- Testing Reservation Agent (Manual Call) ---")
# booking_result = reservation_agent.tools[0]._run(
#     hotel_name="Hotel Paradise", 
#     room_type="standard", 
#     guest_name="Alice Smith", 
#     check_in_date="2026-07-10", 
#     check_out_date="2026-07-15"
# )
# print(f"Reservation Agent Test Result: {booking_result}")

# print("\n--- Testing Modification Agent (Manual Call) ---")
# # Assuming a booking ID from the previous step, e.g., 'BOOK-0001'
# modification_result = modification_agent.tools[0]._run(
#     booking_id="BOOK-0001", 
#     new_check_in="2026-08-01", 
#     action="modify"
# )
# print(f"Modification Agent Test Result: {modification_result}")
